# Task 22 - PCA for High-Cardinality Categoricals (Variance Threshold + Autoencoder before PCA)

**Goal:** reduce dimensionality of high-cardinality categorical features with PCA, using feature selection (**Variance Threshold**) and **autoencoder-based compression** before PCA, so that variance retention is maximised while compute stays small.

**Assumption:** your message was cut off after "minimizing computational ove...". I read it as *computational overhead*.

**What is reduced:** the one-hot expansion of every categorical column with more than 10 levels (`merchant_id`, `product_color`, `product_variation_size_id`, ...) plus a multi-hot matrix of the product **tags** (tags seen on at least `MIN_TAG_COUNT` products).

**Pipelines compared at the same budget of `K_BUDGET` components (held-out variance, runtime, downstream AUC):**
1. PCA on the full one-hot matrix
2. Variance Threshold -> PCA (several thresholds)
3. Variance Threshold -> **Autoencoder** (bottleneck = `K_BUDGET`) -> PCA

**Honest expectation:** one-hot columns are nearly orthogonal, so PCA cannot compress them as well as dense numeric data; the variance curves below show exactly how much is lost at each budget.

**Input:** newest file before this task (missing categoricals borrowed from `data/raw` via `product_id`)
**Outputs:** `data/processed/womens_clothing_pca_features.csv`, `reports/task_22_dimensionality_comparison.csv`, `reports/task_22_pca_variance.csv`, `reports/task_22_analysis_status.csv`

In [ ]:
import re, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

def find_project_root():
    """Walk up from the notebook folder until a folder containing data/processed is found."""
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "data" / "processed").exists():
            return cand
    return start.parent

ROOT = find_project_root()
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
RAW = ROOT / "data" / "raw" / "womens_clothing.csv"
PROC.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def pick(df, *candidates, required=True):
    """Return the real column name matching any candidate (ignores case, spaces, underscores)."""
    lookup = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if _norm(cand) in lookup:
            return lookup[_norm(cand)]
    if required:
        raise KeyError(f"None of {candidates} found. Columns: {list(df.columns)}")
    return None

def load_first(*paths):
    """Load the first existing CSV, so each task builds on the newest previous output."""
    for p in paths:
        if Path(p).exists():
            d = pd.read_csv(p)
            d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
            print(f"Loaded: {Path(p).name}  shape={d.shape}")
            return d, Path(p)
    raise FileNotFoundError("None of these files exist: " + ", ".join(str(p) for p in paths))

def add_from_raw(df, *names):
    """Borrow columns missing from df (e.g. title, tags) from data/raw, matched on product_id."""
    missing = [n for n in names if pick(df, n, required=False) is None]
    if not missing:
        return df
    raw = pd.read_csv(RAW); raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
    kd, kr = pick(df, "product_id", required=False), pick(raw, "product_id", required=False)
    if kd is None or kr is None:
        raise KeyError(f"Need product_id in both files to borrow {missing} from raw")
    got = [pick(raw, n) for n in missing if pick(raw, n, required=False)]
    if not got:
        raise KeyError(f"{missing} not found in raw file either")
    add = raw.drop_duplicates(kr)[[kr] + got].rename(columns={kr: kd})
    out = df.merge(add, on=kd, how="left")
    print("Borrowed from raw:", got)
    return out

# Newest cumulative dataset first; each task reads the newest file produced BEFORE it.
CHAIN = ["womens_clothing_high_cardinality_encoded.csv", "womens_clothing_engagement_features.csv",
         "womens_clothing_sentiment_features.csv", "womens_clothing_age_segmented.csv",
         "womens_clothing_review_text_features.csv", "womens_clothing_market_basket_features.csv",
         "womens_clothing_product_text_features.csv", "womens_clothing_product_tag_features.csv",
         "womens_clothing_product_segmented.csv", "womens_clothing_final.csv"]

# Tasks 19-21 outputs sit in front of the older chain.
LATE = ["womens_clothing_lifecycle_features.csv", "womens_clothing_tfidf_features.csv", "womens_clothing_channel_interaction_features.csv"]
FULL = LATE + CHAIN

# Tasks 22-24 outputs sit in front of everything.
LATE2 = ["womens_clothing_rating_sentiment_encoded.csv", "womens_clothing_pca_features.csv"]
ALL = LATE2 + FULL

import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
LOW_MAX = 10                  # columns with more levels than this are "high cardinality"
MIN_TAG_COUNT = 3
K_BUDGET = 32                 # number of final components (also the autoencoder bottleneck)
VT_GRID = [0.001, 0.003, 0.005, 0.01, 0.02]     # variance = p(1-p); 0.005 ~ level seen on 0.5% of products
AE_HIDDEN, AE_MAX_ITER = 256, 150

## 1. Load data and build the high-dimensional matrix

In [ ]:
df, src = load_first(*[PROC / n for n in ALL[2:]])
df = add_from_raw(df, "merchant_id", "product_color", "product_variation_size_id", "origin_country", "shipping_option_name", "tags")
units_col = pick(df, "units_sold"); tags_col = pick(df, "tags", required=False)
y = (df[units_col] > df[units_col].median()).astype(int).values          # only for the downstream sanity check

NEVER = {"title", "titleorig", "tags", "producturl", "productpicture", "merchantprofilepicture", "merchantinfosubtitle", "productid"}
cat_cols = [c for c in df.columns if (df[c].dtype == object or str(df[c].dtype) in ("string", "category", "str"))
            and re.sub(r"[^a-z]", "", c.lower()) not in NEVER and LOW_MAX < df[c].nunique() < 0.9 * len(df)]
print("High-cardinality categorical columns:", {c: int(df[c].nunique()) for c in cat_cols})

blocks = [pd.get_dummies(df[c].astype("string").fillna("Missing"), prefix=c, dtype=np.uint8) for c in cat_cols]
if tags_col:
    tl = df[tags_col].fillna("").map(lambda s: sorted({t.strip().lower() for t in str(s).split(",") if t.strip()}))
    cnt = pd.Series([t for ts in tl for t in ts]).value_counts(); keep = sorted(cnt[cnt >= MIN_TAG_COUNT].index)
    blocks.append(pd.DataFrame({f"tag_{t}": [int(t in ts) for ts in tl] for t in keep}, index=df.index).astype(np.uint8))
    print(f"Tags: {len(cnt):,} distinct, {len(keep)} kept (>= {MIN_TAG_COUNT} products)")
X = pd.concat(blocks, axis=1); Xv = X.values.astype(np.float32)
print(f"Matrix: {X.shape[0]:,} rows x {X.shape[1]:,} columns | density {Xv.mean():.4f}")

## 2. Helpers: held-out variance, autoencoder, timing

**Original-variance explained (held-out)** = 1 - (reconstruction error on kept columns + variance of dropped columns) / total variance of *all* original columns, all measured on the 20% test split with the training mean. This charges every pipeline for what it throws away, including features removed by the Variance Threshold.

In [ ]:
Xtr, Xte, ytr_, yte_ = train_test_split(Xv, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
mu = Xtr.mean(axis=0); sst_total = float(((Xte - mu) ** 2).sum())

def orig_var_explained(Xte_kept_rec, keep_idx):
    drop = np.setdiff1d(np.arange(Xv.shape[1]), keep_idx)
    sse = float(((Xte[:, keep_idx] - Xte_kept_rec) ** 2).sum()) + float(((Xte[:, drop] - mu[drop]) ** 2).sum())
    return 1 - sse / sst_total

class Autoencoder:
    """Small dense autoencoder built on sklearn's MLPRegressor (input -> H -> K -> H -> input)."""
    def __init__(self, k, hidden=AE_HIDDEN, max_iter=AE_MAX_ITER):
        self.k, self.m = k, MLPRegressor(hidden_layer_sizes=(hidden, k, hidden), activation="relu", solver="adam", max_iter=max_iter,
                                         learning_rate_init=1e-3, alpha=1e-5, random_state=RANDOM_STATE, early_stopping=False)
    def fit(self, A): self.m.fit(A, A); return self
    def encode(self, A):
        relu = lambda z: np.maximum(z, 0)
        h1 = relu(A @ self.m.coefs_[0] + self.m.intercepts_[0]); return relu(h1 @ self.m.coefs_[1] + self.m.intercepts_[1])
    def reconstruct(self, A): return self.m.predict(A)

def downstream_auc(Z):
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.3)), Z, y, cv=cv, scoring="roc_auc").mean()

## 3. Pipeline comparison at a fixed budget

In [ ]:
rows, curves = [], {}

# 1) PCA on everything
t0 = time.time(); p = PCA(n_components=min(K_BUDGET, Xtr.shape[1]), random_state=RANDOM_STATE).fit(Xtr); fit_s = time.time() - t0
rec = p.inverse_transform(p.transform(Xte))
rows.append({"pipeline": "PCA (no selection)", "vt_threshold": 0.0, "n_input_cols": Xv.shape[1], "n_kept_cols": Xv.shape[1], "k": K_BUDGET,
             "heldout_orig_variance_explained": orig_var_explained(rec, np.arange(Xv.shape[1])), "fit_seconds": fit_s})
curves["PCA (no selection)"] = np.cumsum(PCA(n_components=min(100, Xtr.shape[0] - 1), random_state=RANDOM_STATE).fit(Xtr).explained_variance_ratio_)

# 2) Variance Threshold -> PCA
for t in VT_GRID:
    t0 = time.time(); vt = VarianceThreshold(t).fit(Xtr); keep = np.where(vt.get_support())[0]
    if len(keep) <= K_BUDGET: continue
    p = PCA(n_components=K_BUDGET, random_state=RANDOM_STATE).fit(Xtr[:, keep]); fit_s = time.time() - t0
    rec = p.inverse_transform(p.transform(Xte[:, keep]))
    rows.append({"pipeline": f"VarianceThreshold({t}) -> PCA", "vt_threshold": t, "n_input_cols": Xv.shape[1], "n_kept_cols": len(keep), "k": K_BUDGET,
                 "heldout_orig_variance_explained": orig_var_explained(rec, keep), "fit_seconds": fit_s})
comp = pd.DataFrame(rows); comp.round(4)

In [ ]:
# 3) Variance Threshold -> Autoencoder -> PCA (use the VT value that kept the best variance/size trade-off from step 2)
vt_candidates = comp[comp["pipeline"].str.startswith("VarianceThreshold")]
vt_star = float(vt_candidates.sort_values(["heldout_orig_variance_explained"], ascending=False).iloc[0]["vt_threshold"]) if len(vt_candidates) else 0.0
vt = VarianceThreshold(vt_star).fit(Xtr); keep = np.where(vt.get_support())[0]
print(f"Autoencoder input: VT({vt_star}) keeps {len(keep)} of {Xv.shape[1]} columns")
t0 = time.time(); ae = Autoencoder(K_BUDGET).fit(Xtr[:, keep]); ae_s = time.time() - t0
Ztr = ae.encode(Xtr[:, keep]); pca_after = PCA(n_components=K_BUDGET, random_state=RANDOM_STATE).fit(Ztr)
rec_ae = ae.reconstruct(Xte[:, keep])
ae_row = {"pipeline": f"VarianceThreshold({vt_star}) -> Autoencoder -> PCA", "vt_threshold": vt_star, "n_input_cols": Xv.shape[1], "n_kept_cols": len(keep), "k": K_BUDGET,
          "heldout_orig_variance_explained": orig_var_explained(rec_ae, keep), "fit_seconds": ae_s + 0.0}
comp = pd.concat([comp, pd.DataFrame([ae_row])], ignore_index=True)
print(f"Autoencoder final training loss: {ae.m.loss_:.5f} | iterations: {ae.m.n_iter_}")

In [ ]:
# Downstream sanity check: do the compressed features still carry signal? (logistic regression on K components, 5-fold AUC)
def comps_for(row):
    if row["pipeline"].startswith("PCA"): idx = np.arange(Xv.shape[1]); return PCA(K_BUDGET, random_state=RANDOM_STATE).fit_transform(Xv - Xv.mean(0))
    t = row["vt_threshold"]; idx = np.where(VarianceThreshold(t).fit(Xv).get_support())[0]
    if "Autoencoder" in row["pipeline"]:
        a = Autoencoder(K_BUDGET).fit(Xv[:, idx]); return PCA(K_BUDGET, random_state=RANDOM_STATE).fit_transform(a.encode(Xv[:, idx]))
    return PCA(K_BUDGET, random_state=RANDOM_STATE).fit_transform(Xv[:, idx] - Xv[:, idx].mean(0))
comp["downstream_cv_auc"] = [downstream_auc(comps_for(r)) for _, r in comp.iterrows()]
full_auc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.05)), Xv, y, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="roc_auc").mean()
comp["full_onehot_cv_auc"] = full_auc
comp["compression_ratio"] = (comp["n_input_cols"] / comp["k"]).round(1)
comp.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
c = comp.sort_values("heldout_orig_variance_explained")
ax[0].barh(c["pipeline"], c["heldout_orig_variance_explained"], color="#4c72b0"); ax[0].set_title(f"Held-out original variance explained at K={K_BUDGET}")
ax[1].barh(c["pipeline"], c["fit_seconds"], color="#c44e52"); ax[1].set_title("Fit time (s)")
ax[2].barh(c["pipeline"], c["downstream_cv_auc"], color="#55a868"); ax[2].axvline(full_auc, color="k", ls="--", label="full one-hot"); ax[2].legend(); ax[2].set_title("Downstream CV AUC (high sales)")
plt.tight_layout(); plt.show()

## 4. Variance curve and the winning pipeline

The winner is the pipeline with the highest held-out original-variance explained at the same `K_BUDGET`; among pipelines within 1 percentage point of the best, the **fastest** wins (the overhead criterion).

In [ ]:
best = comp["heldout_orig_variance_explained"].max()
winner = comp[comp["heldout_orig_variance_explained"] >= best - 0.01].sort_values("fit_seconds").iloc[0]
print("Winner:", winner["pipeline"], "| variance explained:", round(winner["heldout_orig_variance_explained"], 4), "| fit s:", round(winner["fit_seconds"], 2))
cv_full = curves["PCA (no selection)"]
plt.figure(figsize=(8, 4.5)); plt.plot(range(1, len(cv_full) + 1), cv_full, marker=".")
plt.axvline(K_BUDGET, color="k", ls="--", label=f"K = {K_BUDGET}"); plt.xlabel("components"); plt.ylabel("cumulative explained variance (train)")
plt.title("PCA variance curve on the full one-hot matrix"); plt.legend(); plt.show()
print({k: int(np.argmax(cv_full >= k) + 1) if (cv_full >= k).any() else f">{len(cv_full)}" for k in (0.5, 0.7, 0.8, 0.9)})

## 5. Fit the winner on all rows and create the features

In [ ]:
t = float(winner["vt_threshold"]); idx = np.where(VarianceThreshold(t).fit(Xv).get_support())[0] if t > 0 else np.arange(Xv.shape[1])
if "Autoencoder" in winner["pipeline"]:
    ae_final = Autoencoder(K_BUDGET).fit(Xv[:, idx]); base = ae_final.encode(Xv[:, idx])
    pca_final = PCA(K_BUDGET, random_state=RANDOM_STATE).fit(base); Z = pca_final.transform(base)
else:
    base = Xv[:, idx]; pca_final = PCA(K_BUDGET, random_state=RANDOM_STATE).fit(base); Z = pca_final.transform(base)
Z = pd.DataFrame(Z, index=df.index, columns=[f"pca_cat_{i + 1}" for i in range(Z.shape[1])])
var_tbl = pd.DataFrame({"component": Z.columns, "explained_variance_ratio": pca_final.explained_variance_ratio_, "cumulative": np.cumsum(pca_final.explained_variance_ratio_)})
display(var_tbl.head(10).round(4))
# which original categories load most on the first components? (only meaningful for the non-autoencoder paths)
if "Autoencoder" not in winner["pipeline"]:
    load = pd.DataFrame(pca_final.components_[:3].T, index=X.columns[idx], columns=["PC1", "PC2", "PC3"])
    display(load.abs().idxmax().to_frame("strongest_loading_feature"))

## 6. Save outputs

In [ ]:
out = pd.concat([df, Z], axis=1)
out_path = PROC / "womens_clothing_pca_features.csv"; out.to_csv(out_path, index=False)
comp.round(5).assign(selected=(comp["pipeline"] == winner["pipeline"])).to_csv(REPORTS / "task_22_dimensionality_comparison.csv", index=False)
var_tbl.round(6).to_csv(REPORTS / "task_22_pca_variance.csv", index=False)
status = pd.DataFrame([{"task": "Task 22 - PCA dimensionality reduction", "status": "completed", "run_at": datetime.now().isoformat(timespec="seconds"),
    "input_file": src.name, "rows": len(out), "categorical_columns": ";".join(cat_cols), "input_dimensions": X.shape[1], "k_components": K_BUDGET,
    "selected_pipeline": winner["pipeline"], "heldout_variance_explained": round(winner["heldout_orig_variance_explained"], 4),
    "downstream_auc_selected": round(winner["downstream_cv_auc"], 4), "downstream_auc_full_onehot": round(full_auc, 4),
    "columns_added": K_BUDGET, "output_dataset": out_path.name}])
status.to_csv(REPORTS / "task_22_analysis_status.csv", index=False); status.T

## 7. Key takeaways (fill in after running)

- How much original variance do 32 components keep, and how does that compare with the 70/80/90% milestones above? Low retention is a property of sparse one-hot data, not a bug.
- Did the Variance Threshold cut runtime without hurting held-out variance? Did the autoencoder beat linear PCA at the same dimensionality?
- Does downstream AUC of the compressed features approach the full one-hot AUC?
- `pca_cat_*` are unsupervised (no target used), so they are safe to use as model inputs. Fit the pipeline on training rows only if you move to a strict train/test workflow.